In [ ]:



#lOAD PACKAGES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
import plotly.express as px
import plotly.graph_objects as go  # Add this line

from plotly.subplots import make_subplots
# Import the processing module from the same folder
sys.path.append(os.path.join("..", "scripts", "analysis"))
from processing import filter_demand, transform_to_internal_time, load_solutions, combine_solutions




# import processing 
# from pivottablejs import pivot_ui
G_save = False


In [ ]:


# input_file = 'RTS-GMLC_v2.4.1'
# demand= []
# random_demand = []
# reserve = []
# energy_reserve = []
# demand_ = pd.read_csv(os.path.join("..", "input", input_file, 'uc','Demand.csv'))
# random_demand_ = pd.read_csv(os.path.join("..", "input", input_file, 'ed','random_demand.csv'))
# reserve_ = pd.read_csv(os.path.join("..", "input", input_file, 'uc','Reserve.csv'))
# energy_reserve_ = pd.read_csv(os.path.join("..", "input", input_file, 'uc','Energy reserve.csv')) 
# demand.append(demand_)
# random_demand.append(random_demand_)
# reserve.append(reserve_)
# energy_reserve.append(energy_reserve_)


# demand = pd.concat(demand)
# demand = transform_to_internal_time(demand).set_index(['day', 'hour'])
# random_demand = pd.concat(random_demand)
# random_demand = transform_to_internal_time(random_demand).set_index(['day', 'hour'])
# reserve = pd.concat(reserve)
# reserve = transform_to_internal_time(reserve).set_index(['day', 'hour'])
# energy_reserve = pd.concat(energy_reserve)
# energy_reserve = transform_to_internal_time(energy_reserve)
# energy_reserve = energy_reserve.rename(columns={'i_hour': 'hour_i', 't_hour': 'hour', 'reserve_up_MW':'energy_reserve_up_MWh','reserve_down_MW':'energy_reserve_down_MWh'}).set_index(['day', 'hour_i','hour'])
# random_demand = filter_demand(demand, random_demand, reserve)
# imbalance = random_demand.sub(demand['demand'], axis=0, level=['day','hour'])


In [ ]:
read_files = False
write_files = True
solution_keys = ['dual_variables', 'storage', 'reserve', 'energy_reserve']

if not read_files:
    ss = [
        # {'solution_folder': f"RTS-GMLC_v18.3s", 'model_type' : 'envelope'},
        # {'solution_folder': f"RTS-GMLC_v19.4s", 'model_type' : 'e-reserve'},
        {'solution_folder': f"RTS-GMLC_v32.3s", 'model_type' : 'envelope'},
        {'solution_folder': f"RTS-GMLC_v32.1s", 'model_type' : 'e-reserve'},
    ]
    days = range(1,50)
    s_uc = []
    s_ed = []
    gcd_KPI_adequacy = []
    gcdi_KPI_adequacy = []
    

    for sol in ss:
        # ρ = sol['ρ']
        s = sol['solution_folder']
        # s_uc_name = 's_uc' if sol['model_type'] == 'stochastic' else 's_uc'
        # s_ed_name = 's_sed'
        s_uc_ = load_solutions("s_uc", os.path.join("..", "output", s), days, solution_keys = solution_keys,  model_type = sol['model_type'], solution_id = s)
        if sol['model_type'] != 'stochastic':
            s_ed_ = load_solutions("s_ed", os.path.join("..", "output", s), days, solution_keys = solution_keys, model_type = sol['model_type'], solution_id = s)
        else:
            s_ed_ = load_solutions("s_suc", os.path.join("..", "output", s), days, solution_keys = solution_keys, model_type = sol['model_type'], solution_id = s)
        s_uc.append(s_uc_)
        s_ed.append(s_ed_)

        # gcd_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcd_KPI_adequacy.parquet"))
        # gcd_KPI_adequacy_ = add_fields(gcd_KPI_adequacy_, model_type = sol['model_type'], ρ=ρ, solution_id = s) 

        # gcdi_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcdi_KPI_adequacy.parquet"))
        # gcdi_KPI_adequacy_ = add_fields(gcdi_KPI_adequacy_, model_type = sol['model_type'], ρ=ρ, solution_id = s)

        # gcd_KPI_adequacy.append(gcd_KPI_adequacy_)
        # gcdi_KPI_adequacy.append(gcdi_KPI_adequacy_)

    s_uc = combine_solutions(s_uc)
    s_ed = combine_solutions(s_ed)
    # gcd_KPI_adequacy = pd.concat(gcd_KPI_adequacy)
    # gcdi_KPI_adequacy = pd.concat(gcdi_KPI_adequacy)

    for k,v in s_uc.items():
        if 'µ' in v.columns:
            s_uc[k]['model_type'] =  v.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)
    for k,v in s_ed.items():
        if 'µ' in v.columns:
            s_ed[k]['model_type'] = v.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)
    # if 'µ' in gcdi_KPI_adequacy.columns: 
    #     gcdi_KPI_adequacy['model_type'] = gcdi_KPI_adequacy.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)
    #     gcd_KPI_adequacy['model_type'] = gcd_KPI_adequacy.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)
    if write_files:
        for k,v in s_uc.items():
            v.to_csv(f's_uc_{k}.csv', index=False)
        for k,v in s_ed.items():
            v.to_csv(f's_ed_{k}.csv', index=False)

else:
    s_uc = {}
    s_ed = {}
    for solution_key in solution_keys:
        s_uc[solution_key] = pd.read_csv(f's_uc_{solution_key}.csv')
        # s_ed[solution_key] = pd.read_csv(f's_ed_{solution_key}.csv')    

    # gcd_KPI_adequacy = pd.read_csv('gcd_KPI_adequacy.csv', index_col=0)
    # gcdi_KPI_adequacy = pd.read_csv('gcdi_KPI_adequacy.csv', index_col=0)







In [ ]:
envelope_dual = s_uc['dual_variables'][[
    'hour','hour_i','day','r_id','model_type',
    'dual_SOE_up_max_MU_MW', 'dual_SOE_down_max_MU_MW', 'dual_SOE_up_min_MU_MW', 'dual_SOE_down_min_MU_MW',
    'dual_ESOE_up_max_MU_MW', 'dual_ESOE_down_max_MU_MW', 'dual_ESOE_up_min_MU_MW', 'dual_ESOE_down_min_MU_MW',
    ]].copy()
storage_list = range(101,131,1)# We filter to reduce df size.
envelope_dual = envelope_dual[envelope_dual['r_id'].isin(storage_list)]
# envelope_dual['hour'] = envelope_dual['hour'] - (envelope_dual['day']-1)*24


In [ ]:
# envelope_dual.to_csv('envelope_dual.csv')

In [ ]:
storage_activation = s_uc['storage'][['hour','day','r_id','model_type','charge_MW', 'discharge_MW']]
storage_activation.dropna(subset = ['charge_MW', 'discharge_MW'], inplace = True)
# fields = ['charge_MW', 'discharge_MW']
# for f in fields:
#     storage_activation[f] = 1*(storage_activation[f].abs() > 0.000001)


In [ ]:
energy_reserve_activation = s_uc['energy_reserve'][['hour','hour_i','day','r_id','model_type','energy_reserve_up_MW', 'energy_reserve_down_MW']] 
energy_reserve_activation = energy_reserve_activation[energy_reserve_activation['r_id'].isin(storage_list)]

In [ ]:
fields = [
    'dual_SOE_up_max_MU_MW',
    'dual_SOE_down_max_MU_MW',
    'dual_SOE_up_min_MU_MW',
    'dual_SOE_down_min_MU_MW',
    'dual_ESOE_up_max_MU_MW',
    'dual_ESOE_down_max_MU_MW',
    'dual_ESOE_up_min_MU_MW',
    'dual_ESOE_down_min_MU_MW']
activation = envelope_dual.copy()
for f in fields:
    activation[f] = 1*(activation[f].abs() > 0.000001)
activation = pd.merge(activation, storage_activation, how = 'left', on = ['hour','day', 'r_id', 'model_type'])
# energy_reserve['model_type'] = 'e-reserve'
# activation = pd.merge(activation, energy_reserve, how = 'left', on = ['hour', 'hour_i','day', 'model_type'])
# energy_reserve['model_type'] = 'e-reserve'
# activation = pd.merge(activation, reserve, how = 'left', on = ['hour','day'])
activation =  pd.merge(activation, energy_reserve_activation, how = 'left', on = ['hour','hour_i','day', 'r_id', 'model_type'])

In [ ]:
activation

In [ ]:
activation.groupby(['model_type'])[fields].sum()

In [ ]:
activation.groupby(['day', 'r_id','model_type'])[fields].count()

In [ ]:
activation.groupby(['day', 'r_id','model_type'])[fields].sum()

In [ ]:
field_to_plot = 'dual_ESOE_up_max_MU_MW'

count_activation_by_reserve = activation[activation.model_type == 'e-reserve'].copy()
count_activation_by_reserve['reserve_up'] = (count_activation_by_reserve['energy_reserve_up_MW']*count_activation_by_reserve[field_to_plot])>0.00001
count_activation_by_reserve['reserve_down'] = (count_activation_by_reserve['energy_reserve_down_MW']*count_activation_by_reserve[field_to_plot])>0.00001
count_activation_by_reserve = count_activation_by_reserve.groupby(['r_id','day', 'hour', 'hour_i']).sum().reset_index()

In [ ]:
activation[activation.model_type == 'e-reserve'].groupby(['day']).count()

In [ ]:
stacked = count_activation_by_reserve.groupby(['hour']).sum().reset_index().melt(
    id_vars=['hour'],
    value_vars=['reserve_up', 'reserve_down'],
    # var_name='action_type',
    # value_name='action_value'
)
fig = px.bar(stacked, x = 'hour', y = 'value', color = 'variable', barmode='group')
fig.update_layout(yaxis_title='number of binding constraints')
fig.show()

In [ ]:

field_to_plot = 'dual_ESOE_up_max_MU_MW'
count_fields_by_charge = activation.copy()
count_fields_by_charge['charging'] = (count_fields_by_charge['charge_MW']*count_fields_by_charge[field_to_plot])>0
count_fields_by_charge['discharging'] = (count_fields_by_charge['discharge_MW']*count_fields_by_charge[field_to_plot])>0
count_fields_by_charge['standby'] = ((count_fields_by_charge['charge_MW'] == 0)*(count_fields_by_charge['discharge_MW'] == 0)*count_fields_by_charge[field_to_plot])>0
count_fields_by_charge = count_fields_by_charge.groupby(['r_id','day', 'hour', 'hour_i']).sum()

In [ ]:
stacked = count_fields_by_charge.groupby(['hour']).sum().reset_index().melt(
    id_vars=['hour'],
    value_vars=['charging', 'discharging', 'standby'],
    # var_name='action_type',
    # value_name='action_value'
)
fig = px.bar(stacked, x = 'hour', y = 'value', color = 'variable', barmode='group')
fig.update_layout(yaxis_title='number of binding constraints')
fig.show()

In [ ]:
count_activation = activation[activation.model_type == 'e-reserve'].groupby(['r_id','day','hour_i', 'hour']).sum()
count_activation.replace(0, np.nan, inplace=True)
count_activation.dropna(axis = 0, how = 'all', subset = fields, inplace = True)

px.scatter(count_activation.groupby(['hour', 'hour_i']).sum().reset_index(), x = 'hour', y = 'hour_i', color = field_to_plot)

In [ ]:
px.bar(count_activation.groupby('hour').sum().reset_index(), x = 'hour', y = field_to_plot, color = field_to_plot)

In [ ]:

# field_to_plot = 'discharge_action'
# fig = px.bar(count_fields_by_charge.groupby(['hour']).sum().reset_index(), x = 'hour', y = field_to_plot, color = field_to_plot)
# fig.update_layout(yaxis_title='frequency of binding')
# fig.show()

In [ ]:

# field_to_plot = 'charge_action'
# px.scatter(count_fields_by_charge.groupby(['hour', 'hour_i']).sum().reset_index(), x = 'hour', y = 'hour_i', color = field_to_plot)


In [ ]:
# field_to_plot = 'discharge_action'
# px.scatter(count_fields_by_charge.groupby(['hour', 'hour_i']).sum().reset_index(), x = 'hour', y = 'hour_i', color = field_to_plot)

In [ ]:
# field_to_plot = 'dual_ESOE_up_max_MU_MW'
# px.scatter(count_fields.loc[101,50,:,:].reset_index(), x = 'hour', y = 'hour_i', color = field_to_plot)

In [ ]:
relative_activation_daily = activation[activation.model_type == 'e-reserve'].groupby(['day']).sum()[field_to_plot]/activation[activation.model_type == 'e-reserve'].groupby(['day']).count()[field_to_plot]*100
y_field = 'activation_frequency [%]'
px.bar(relative_activation_daily.reset_index().rename(columns={field_to_plot: y_field}), x = 'day', y = y_field, color = y_field)

In [ ]:
field_to_plot = 'dual_ESOE_up_max_MU_MW'
px.bar(count_activation.groupby(['r_id']).sum().reset_index(), x = 'r_id', y = field_to_plot, color = field_to_plot)

In [ ]:

# px.scatter(count_activation.groupby(['r_id', 'hour', 'hour_i']).sum().reset_index(), x = 'hour', y = 'hour_i', facet_col = 'r_id', color = field_to_plot)

In [ ]:
activation

In [ ]:
field_to_plot = 'charging'
px.bar(count_fields_by_charge.groupby(['hour']).sum().reset_index(), x = 'hour', y = field_to_plot, color = field_to_plot)

In [ ]:
activation_stacked = activation.melt(
    id_vars=['hour', 'hour_i', 'day', 'r_id', 'model_type', 
             'dual_SOE_up_max_MU_MW', 'dual_SOE_down_max_MU_MW', 
             'dual_SOE_up_min_MU_MW', 'dual_SOE_down_min_MU_MW',
             'dual_ESOE_up_max_MU_MW', 'dual_ESOE_down_max_MU_MW', 
             'dual_ESOE_up_min_MU_MW', 'dual_ESOE_down_min_MU_MW'],
    value_vars=['energy_reserve_up_MW', 'energy_reserve_down_MW'],
    var_name='reserve_type',
    value_name='energy_reserve_MWh'
)
# filter = (activation_stacked.model_type == 'e-reserve') & (activation_stacked.dual_ESOE_up_max_MU_MW == 1)
fig = px.box(activation_stacked, x='hour', y='energy_reserve_MWh', color = 'reserve_type', facet_col = 'dual_ESOE_up_max_MU_MW')
# fig.update_yaxes(matches=None) 
# fig.for_each_yaxis(lambda yaxis: yaxis.update(showticklabels=True))
fig.show()

In [ ]:

activation_stacked = activation.melt(
    id_vars=['hour', 'hour_i', 'day', 'r_id', 'model_type', 
             'dual_SOE_up_max_MU_MW', 'dual_SOE_down_max_MU_MW', 
             'dual_SOE_up_min_MU_MW', 'dual_SOE_down_min_MU_MW',
             'dual_ESOE_up_max_MU_MW', 'dual_ESOE_down_max_MU_MW', 
             'dual_ESOE_up_min_MU_MW', 'dual_ESOE_down_min_MU_MW'],
    value_vars=['reserve_up_MW', 'reserve_down_MW'],
    var_name='reserve_type',
    value_name='reserve_MW'
)
filter = (activation_stacked.model_type == 'e-reserve')
fig = px.box(activation_stacked[filter], x='hour', y='reserve_MW', color = 'reserve_type', facet_col = 'dual_ESOE_up_max_MU_MW')
# fig.update_yaxes(matches=None) 
# fig.for_each_yaxis(lambda yaxis: yaxis.update(showticklabels=True))
fig.show()

In [ ]:
# activation_stacked = activation.melt(
#     id_vars=['hour', 'hour_i', 'day', 'r_id', 'model_type', 
#              'dual_SOE_up_max_MU_MW', 'dual_SOE_down_max_MU_MW', 
#              'dual_SOE_up_min_MU_MW', 'dual_SOE_down_min_MU_MW',
#              'dual_ESOE_up_max_MU_MW', 'dual_ESOE_down_max_MU_MW', 
#              'dual_ESOE_up_min_MU_MW', 'dual_ESOE_down_min_MU_MW'],
#     value_vars=['charge_MW', 'discharge_MW'],
#     var_name='operation_type',
#     value_name='power_MW'
# )
# filter = (activation_stacked.model_type == 'e-reserve')
# fig = px.box(activation_stacked[filter], x='hour', y='power_MW', facet_row='operation_type', color = 'operation_type', facet_col = 'dual_ESOE_up_max_MU_MW', hover_data = ['r_id', 'day', 'hour_i'])
# fig.update_yaxes(matches=None) 
# fig.update_layout(
#     autosize=False,
#     width=1200,
#     height=800,
# )
# fig.for_each_yaxis(lambda yaxis: yaxis.update(showticklabels=True))